In [0]:

# %sql 
# CREATE CATALOG IF NOT EXISTS weather_catalog 
# MANAGED LOCATION 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/weather_catalog';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5954769199729148>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "\n\nCREATE CATALOG IF NOT EXISTS weather_catalog \nMANAGED LOCATION 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/weather_catalog';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:183, in Sql

In [0]:
# %sql
# CREATE SCHEMA IF NOT EXISTS weather_catalog.gold;

In [0]:
# %sql

# CREATE TABLE IF NOT EXISTS weather_catalog.gold.location_master
# USING delta
# LOCATION 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/location_master/';

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6096282399629170>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "\nCREATE TABLE IF NOT EXISTS weather_catalog.gold.location_master\nUSING delta\nLOCATION 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/location_master/';\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sq

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table(
    "weather_catalog.silver.location_master"
)

#display(silver_df)

EnglishName,Key,GeoPosition_Latitude,GeoPosition_Longitude
Mumbai,204842,19.143,72.878
Delhi,202396,28.643,77.118
Pune,204848,18.499,73.866


In [0]:
from pyspark.sql.functions import col

gold_df = (
    silver_df
    .select(
        col("Key").alias("location_key"),
        col("EnglishName").alias("city_name"),
        col("GeoPosition_Latitude").alias("latitude"),
        col("GeoPosition_Longitude").alias("longitude")
    )
    .dropDuplicates(["location_key"])
)

In [0]:
print("Gold Count:", gold_df.count())

#display(gold_df)

Gold Count: 3


location_key,city_name,latitude,longitude
204842,Mumbai,19.143,72.878
202396,Delhi,28.643,77.118
204848,Pune,18.499,73.866


Write Gold Table

In [0]:
# gold_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option(
#         "path",
#         "abfss://gold@stweatherprojectnew.dfs.core.windows.net/gold/location_master"
#     ) \
#     .saveAsTable("weather_catalog.gold.location_master")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5954769199729158>, line 8
      1 gold_df.write \
      2     .format("delta") \
      3     .mode("overwrite") \
      4     .option(
      5         "path",
      6         "abfss://gold@stweatherprojectnew.dfs.core.windows.net/gold/location_master"
      7     ) \
----> 8     .saveAsTable("weather_catalog.gold.location_master")

File /databricks/spark/python/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/spark/python/pyspark/sql/connect/client/core.py:1589, in SparkConnec

In [0]:
# gold_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .saveAsTable("weather_catalog.gold.location_master")

In [0]:
# %sql
# DESCRIBE DETAIL weather_catalog.gold.location_master;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f287adfa-3266-4f01-86b0-69e8d9d16362,weather_catalog.gold.location_master,null,abfss://silver@stweatherprojectnew.dfs.core.windows.net/silver/location_master,2026-06-25T18:19:46.752Z,2026-06-26T04:01:38Z,List(),List(),1,1567,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03_location_master_silver_to_gold
# MAGIC **Purpose:** Read data from the Silver layer, ensure Gold namespaces and storage environments exist, and write to the Gold layer in Unity Catalog.

# COMMAND ----------
# 1. Imports & Configuration
from pyspark.sql import functions as F

# Define paths and catalog variables
silver_path = "abfss://silver@stweatherprojectnew.dfs.core.windows.net/location_master"
gold_path   = "abfss://gold@stweatherprojectnew.dfs.core.windows.net/gold/location_master"

catalog_name = "weather_catalog"
schema_name  = "gold"
table_name   = "location_master"
full_table_identifier = f"{catalog_name}.{schema_name}.{table_name}"

# COMMAND ----------
# 2. Read Cleaned Data from Silver Layer
print(f"Reading data from Silver path: {silver_path}")
silver_df = spark.read.format("delta").load(silver_path)

# COMMAND ----------
# 3. Gold Layer Infrastructure Setup (Unity Catalog)

# Ensure the parent Catalog exists
print("Creating/Verifying Gold External Location...")
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS gold_storage_location
URL 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/'
WITH (CREDENTIAL weather_credential);
""")

# STEP 2: Create the parent Catalog using a path covered by the External Location above
print(f"Creating Catalog with Managed Location: {catalog_name}")
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {catalog_name}
MANAGED LOCATION 'abfss://gold@stweatherprojectnew.dfs.core.windows.net/catalog_root/'
""")

# STEP 3: Create the Gold Schema (Database) inside the newly created Catalog
print(f"Creating Schema if not exists: {catalog_name}.{schema_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# COMMAND ----------
# 4. Optional Transformation / Filtering 
# (Example: Filtering for specific target nodes or aggregation if necessary)
# For now, passing curated data directly to the reporting tier
gold_df = silver_df

# COMMAND ----------
# 5. Write to Gold Managed External Table
# 1. Drop the existing table metadata pointing to the incorrect (silver) location
print(f"Dropping stale table metadata for: {full_table_identifier}")
spark.sql(f"DROP TABLE IF EXISTS {full_table_identifier}")

# 2. Write the table to the correct Gold location
print(f"Writing final data to Gold Delta table: {full_table_identifier}")
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", gold_path) \
    .saveAsTable(full_table_identifier)

# COMMAND ----------
# 6. Verify Final Dataset
print("\n--- Pipeline Successfully Completed ---")
print(f"Total Rows Written to Gold: {gold_df.count()}")

# Display sample output
display(spark.table(full_table_identifier).limit(10))

Reading data from Silver path: abfss://silver@stweatherprojectnew.dfs.core.windows.net/location_master
Creating/Verifying Gold External Location...
Creating Catalog with Managed Location: weather_catalog
Creating Schema if not exists: weather_catalog.gold
Dropping stale table metadata for: weather_catalog.gold.location_master
Writing final data to Gold Delta table: weather_catalog.gold.location_master

--- Pipeline Successfully Completed ---
Total Rows Written to Gold: 3


EnglishName,Key,GeoPosition_Latitude,GeoPosition_Longitude
Mumbai,204842,19.143,72.878
Delhi,202396,28.643,77.118
Pune,204848,18.499,73.866
